<a href="https://colab.research.google.com/github/itsFr4nc0/red-neuronal-creditos/blob/main/Modelo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score, classification_report

import torch
import torch.nn as nn
import torch.optim as optim

In [33]:
df = pd.read_csv("loan.csv", low_memory=False)
df.head()

,id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,int_rate,installment,grade,sub_grade,...,total_bal_il,il_util,open_rv_12m,open_rv_24m,max_bal_bc,all_util,total_rev_hi_lim,inq_fi,total_cu_tl,inq_last_12m
0,1077501,1296599,5000.0,5000.0,4975.0,36 months,10.65,162.87,B,B2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1077430,1314167,2500.0,2500.0,2500.0,60 months,15.27,59.83,C,C4,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1077175,1313524,2400.0,2400.0,2400.0,36 months,15.96,84.33,C,C5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1076863,1277178,10000.0,10000.0,10000.0,36 months,13.49,339.31,C,C1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1075358,1311748,3000.0,3000.0,3000.0,60 months,12.69,67.79,B,B5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 887379 entries, 0 to 887378
Data columns (total 74 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   id                           887379 non-null  int64  
 1   member_id                    887379 non-null  int64  
 2   loan_amnt                    887379 non-null  float64
 3   funded_amnt                  887379 non-null  float64
 4   funded_amnt_inv              887379 non-null  float64
 5   term                         887379 non-null  object 
 6   int_rate                     887379 non-null  float64
 7   installment                  887379 non-null  float64
 8   grade                        887379 non-null  object 
 9   sub_grade                    887379 non-null  object 
 10  emp_title                    835917 non-null  object 
 11  emp_length                   842554 non-null  object 
 12  home_ownership               887379 non-null  object 
 13 

In [35]:
cols_to_drop = [
    "total_pymnt",
    "total_pymnt_inv",
    "total_rec_prncp",
    "total_rec_int",
    "total_rec_late_fee",
    "recoveries",
    "collection_recovery_fee",
    "last_pymnt_amnt",
    "out_prncp",
    "out_prncp_inv"
]

cols_drop_extra = [
    "id",
    "member_id",
    "url",
    "desc",
    "title",
    "issue_d"
]

In [36]:
# eliminar leakage
df = df.drop(columns=cols_to_drop)

# eliminar extras
df = df.drop(columns=cols_drop_extra)

# eliminar columnas con muchos nulos
threshold = 0.1 * len(df)
df = df.dropna(thresh=threshold, axis=1)

In [37]:
df["term"] = df["term"].str.extract(r"(\d+)").astype(float)

In [38]:
df["emp_length"] = df["emp_length"].str.extract(r"(\d+)")
df["emp_length"] = pd.to_numeric(df["emp_length"], errors="coerce")

In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 887379 entries, 0 to 887378
Data columns (total 41 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   loan_amnt                    887379 non-null  float64
 1   funded_amnt                  887379 non-null  float64
 2   funded_amnt_inv              887379 non-null  float64
 3   term                         887379 non-null  float64
 4   int_rate                     887379 non-null  float64
 5   installment                  887379 non-null  float64
 6   grade                        887379 non-null  object 
 7   sub_grade                    887379 non-null  object 
 8   emp_title                    835917 non-null  object 
 9   emp_length                   842554 non-null  float64
 10  home_ownership               887379 non-null  object 
 11  annual_inc                   887375 non-null  float64
 12  verification_status          887379 non-null  object 
 13 

In [40]:
print(df["loan_status"].unique())

['Fully Paid' 'Charged Off' 'Current' 'Default' 'Late (31-120 days)'
 'In Grace Period' 'Late (16-30 days)'
 'Does not meet the credit policy. Status:Fully Paid'
 'Does not meet the credit policy. Status:Charged Off' 'Issued']


In [41]:
def map_loan_status(status):
    if status in ["Fully Paid", "Does not meet the credit policy. Status:Fully Paid"]:
        return 0
    elif status in ["Charged Off", "Default", "Late (31-120 days)",
                    "Does not meet the credit policy. Status:Charged Off"]:
        return 1
    else:
        return np.nan

df["target"] = df["loan_status"].apply(map_loan_status)

df = df.dropna(subset=["target"])

In [42]:
df = df.drop(columns=["loan_status"])

In [44]:
cols_leakage = [
    "last_pymnt_d",
    "next_pymnt_d",
    "last_credit_pull_d"
]

cols_text = [
    "emp_title",
    "zip_code"
]

In [45]:
df = df.drop(columns=cols_leakage + cols_text + ["earliest_cr_line"])

In [47]:
for col in df.select_dtypes(include=["float64"]).columns:
    df[col] = df[col].fillna(df[col].median())

for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].fillna("Unknown")

In [48]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 268530 entries, 0 to 887371
Data columns (total 35 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   loan_amnt                    268530 non-null  float64
 1   funded_amnt                  268530 non-null  float64
 2   funded_amnt_inv              268530 non-null  float64
 3   term                         268530 non-null  float64
 4   int_rate                     268530 non-null  float64
 5   installment                  268530 non-null  float64
 6   grade                        268530 non-null  object 
 7   sub_grade                    268530 non-null  object 
 8   emp_length                   268530 non-null  float64
 9   home_ownership               268530 non-null  object 
 10  annual_inc                   268530 non-null  float64
 11  verification_status          268530 non-null  object 
 12  pymnt_plan                   268530 non-null  object 
 13  purp

In [62]:
X = df.drop(columns=["target"])
y = df["target"]

In [63]:
num_cols = X.select_dtypes(include=["float64"]).columns
cat_cols = X.select_dtypes(include=["object"]).columns

In [64]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

In [65]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [66]:
X_train = preprocessor.fit_transform(X_train)
X_test = preprocessor.transform(X_test)

In [67]:
X_train = X_train.toarray()
X_test = X_test.toarray()

In [68]:
import torch

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1,1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1,1)

In [69]:
device = "cuda" if torch.cuda.is_available() else "cpu"

X_train_tensor = X_train_tensor.to(device)
X_test_tensor = X_test_tensor.to(device)
y_train_tensor = y_train_tensor.to(device)
y_test_tensor = y_test_tensor.to(device)

In [70]:
import torch.nn as nn

class RedRiesgo(nn.Module):
    def __init__(self, input_dim):
        super().__init__()

        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.2),

            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.model(x)

In [71]:
input_dim = X_train_tensor.shape[1]

model = RedRiesgo(input_dim).to(device)

import torch.optim as optim

# Calculo un peso para que mejore el modelo (Me salia que todos eran buenos pagadores)
pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([pos_weight]).to(device)
)

optimizer = optim.Adam(model.parameters(), lr=0.001)

In [74]:
epochs = 50

for epoch in range(epochs):
    model.train()

    optimizer.zero_grad()

    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    loss.backward()
    optimizer.step()

    if epoch % 5 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item():.4f}")

Epoch 0, Loss: 0.9910
Epoch 5, Loss: 0.9896
Epoch 10, Loss: 0.9870
Epoch 15, Loss: 0.9830
Epoch 20, Loss: 0.9828
Epoch 25, Loss: 0.9820
Epoch 30, Loss: 0.9797
Epoch 35, Loss: 0.9789
Epoch 40, Loss: 0.9785
Epoch 45, Loss: 0.9782


In [75]:
from sklearn.metrics import roc_auc_score, classification_report

model.eval()

with torch.no_grad():
    logits = model(X_test_tensor)
    probs = torch.sigmoid(logits)

    y_pred = (probs > 0.5).float()

y_test_np = y_test_tensor.cpu().numpy()
probs_np = probs.cpu().numpy()

print(classification_report(y_test_np, y_pred.cpu().numpy()))
print("AUC:", roc_auc_score(y_test_np, probs_np))

              precision    recall  f1-score   support

         0.0       0.87      0.63      0.73     41942
         1.0       0.34      0.67      0.45     11764

    accuracy                           0.64     53706
   macro avg       0.61      0.65      0.59     53706
weighted avg       0.76      0.64      0.67     53706

AUC: 0.7080773124366575
